In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "adlsjpccd01";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-gold`
URL 'abfss://gold@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
-- DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.gold;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://gold@{storageName}.dfs.core.windows.net/",True)

In [0]:
-- CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.gold;

###Tablas Bronze

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.customers (
customer_id string,
customer_unique_id string,
customer_zip_code_prefix string,
customer_city string,
customer_state string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/customers"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.geolocation (
geolocation_zip_code_prefix integer,
geolocation_lat double,
geolocation_lng double,
geolocation_city string,
geolocation_state string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/geolocation"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.items (
order_id string,
order_item_id integer,
product_id string, 
seller_id string,
shipping_limit_date timestamp,
price double,
freight_value double,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/items"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.payments (
order_id string,
payment_sequential integer,
payment_type string,
payment_installments integer,
payment_value double,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/payments"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.orders (
order_id string,
customer_id	string,
order_status string,
order_purchase_timestamp timestamp,
order_approved_at timestamp,
order_delivered_carrier_date timestamp,
order_delivered_customer_date timestamp,
order_estimated_delivery_date timestamp,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/orders"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.products (
product_id string,
product_category_name string,
product_name_lenght integer,
product_description_lenght integer,
product_photos_qty integer,
product_weight_g integer,
product_length_cm integer,
product_height_cm integer,
product_width_cm integer,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/products"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.sellers (
seller_id string,
seller_zip_code_prefix string,
seller_city string,
seller_state string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/sellers"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.translation (
product_category_name string,
product_category_name_english string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/translation"

###Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.silver.orders_transformed (
    order_id                        string,
    customer_id                     string,
    customer_unique_id              string,
    customer_zip_code_prefix        string,
    customer_city                   string,
    customer_state                  string,
    order_status                    string,
    order_purchase_timestamp        timestamp,
    order_delivered_customer_date   timestamp,
    order_estimated_delivery_date   timestamp,
    payment_type                    string,
    payment_installments            integer,
    payment_value                   double,
    product_id                      string,
    product_category_name           string,
    product_category_name_english   string,
    order_item_id                   integer,
    seller_id                       string,
    shipping_limit_date              timestamp,
    price                           double,
    freight_value                   double,
    seller_zip_code_prefix          string,
    seller_city                     string,
    seller_state                    string,
    geolocation_zip_code_prefix     integer,
    geolocation_lat                 double,
    geolocation_lng                 double,
    geolocation_city                string,
    geolocation_state               string,
    ingestion_date                  timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/orders_transformed";

###Tablas Golden

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.gold.gold_sales_monthly (
    order_month         timestamp,
    category            string,
    num_orders          bigint,
    revenue             double,
    revenue_ex_freight  double,
    avg_order_value     double,
    mom_growth_pct      double
)
USING DELTA
LOCATION "abfss://gold@${storageName}.dfs.core.windows.net/gold_sales_monthly";

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.gold.gold_delivery_performance (
    category                    string,
    customer_state              string,
    num_orders                  bigint,
    pct_late                    double,
    avg_delay_days              double,
    avg_delay_days_when_late    double
)
USING DELTA
LOCATION "abfss://gold@${storageName}.dfs.core.windows.net/gold_delivery_performance";